# Baselines: Primärverband (Lohmann & Rauscher)
Random- und Majority-Baseline für die Primärverband-Empfehlung, **getrennt pro Experte** auf **Level 1 (Produkt)** und **Level 2 (Produktfamilie)**.

Die Berechnung liegt in `primaerverband_baselines.py`. Sie nutzt **dieselbe Bewertung wie die KI-Scores** in `primaerverband_calc.py` (Best-Path-F1, `map_level_2`, gleiche Ausschlussregeln). Damit sind die Werte direkt mit `calculate_primaerverband_lr_expert_scores()` vergleichbar.

## Datengrundlage und Bewertung
- **Ground Truth:** `Experte1_LR_GroundTruth_normalised.csv`, `Experte2_LR_GroundTruth_normalised.csv` (Präferenz + optionale Alternative)
- **Produktpool:** die 32 Primärverbände aus `data/l&r_produktkatalog/lr1_produktkatalog.md`
- **Metrik:** Best-Path-F1 = höchster Set-F1 über alle Kombinationen aus (Präferenz | Alternative) der Vorhersage × (Präferenz | Alternative) des Experten, in %
- **Ausschluss:** Wunden, an denen der Experte keine Empfehlung abgegeben hat. Daraus folgt **n = 57** (Experte 1) bzw. **n = 58** (Experte 2).
- **Level 2:** Produkte werden vor der Bewertung mit `map_level_2` auf ihre Familie gemappt (z. B. P Sensitive und P + PHMB werden beide zu *Suprasorb P*).

In [1]:
import os, sys
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.insert(0, BASE_DIR)
sys.path.insert(0, os.path.join(BASE_DIR, "notebooks", "analyse_categories"))

import pandas as pd
from primaerverband_baselines import (
    load_catalog_products, load_expert_sets, evaluable_wounds,
    random_baseline_exact, majority_baseline_loo, compute_baseline_table,
)
import primaerverband_calc as pv

print("Produkte im Katalog:", len(load_catalog_products()))
for e in (1, 2):
    print(f"Experte {e}: {len(evaluable_wounds(load_expert_sets(e)))} bewertbare Wunden")

Produkte im Katalog: 32
Experte 1: 57 bewertbare Wunden
Experte 2: 58 bewertbare Wunden


## Random-Baseline

**Idee:** Wie gut wäre eine Empfehlung, die rein zufällig Produkte aus dem Katalog wählt?

**Vorgehen pro Wunde:**
1. Ein zufälliges **Präferenz-Set** und ein zufälliges **Alternativ-Set** werden aus den 32 Primärverbänden gezogen, ohne Zurücklegen innerhalb eines Sets und beide unabhängig voneinander.
2. Die **Set-Größen entsprechen denen der KI** (Two-Stage) an derselben Wunde. Nennt die KI z. B. 2 Produkte in der Präferenz und 1 in der Alternative, zieht auch der Zufall 2 + 1. So hat der Zufall dieselbe Anzahl an „Versuchen“ wie die KI, denn größere Sets bzw. eine zusätzliche Alternative erhöhen den Best-Path-F1. (Im Mittel nennt die KI 1,9 Produkte in der Präferenz und 1,8 in der Alternative, eine Alternative gibt es bei allen 60 Wunden.)
3. Bewertung mit dem Best-Path-F1 gegen den jeweiligen Experten.

**Exakter Erwartungswert statt Simulation:** Es wird nicht zufällig gezogen, sondern **alle möglichen Sets** werden aufgezählt (z. B. 496 Möglichkeiten für 2 aus 32). Für jedes mögliche Set wird der beste F1 gegen das Präferenz- oder Alternativ-Set des Experten berechnet: $U$ für das Präferenz-Set, $V$ für das Alternativ-Set. Weil beide Sets unabhängig gezogen werden und der Best-Path-F1 das Maximum über beide ist, gilt pro Wunde

$$E[\mathrm{F1}] = E[\max(U, V)],$$

was sich exakt aus den Verteilungen von $U$ und $V$ berechnen lässt. Das Ergebnis ist ein fester, reproduzierbarer Wert ohne Zufallsschwankung. (Eine Simulation mit 2000 Durchläufen ergab dieselben Werte auf ±0,1.)

In [2]:
rows = []
for level in (1, 2):
    for e in (1, 2):
        r = random_baseline_exact(e, level)
        rows.append({"Level": level, "Experte": e, "n": r["n"], "Random F1 (%)": round(r["f1"], 1)})
pd.DataFrame(rows)

,Level,Experte,n,Random F1 (%)
0,1,1,57,11.5
1,1,2,58,17.3
2,2,1,57,23.5
3,2,2,58,35.1


## Majority-Baseline (Leave-One-Out)

**Idee:** Wie gut wäre eine Empfehlung, die immer das vorschlägt, was der Experte am häufigsten wählt?

**Vorgehen pro Wunde:**
1. Aus den **übrigen Wunden** desselben Experten (Leave-One-Out, die aktuelle Wunde wird nicht mitgezählt) wird die **häufigste Präferenzkombination** bestimmt.
2. Diese Kombination wird als Vorhersage verwendet, **ohne Alternative**.
3. Bewertung mit dem Best-Path-F1 gegen denselben Experten.

**Warum vollständige Kombinationen und nicht die häufigsten Einzelprodukte?** Die zwei häufigsten Einzelprodukte können eine Kombination ergeben, die der Experte **nie** gemeinsam einsetzt. Bei Experte 1 wären das *Suprasorb CNP* (18×) und *P Sensiflex* (16×), die in keiner einzigen Präferenz zusammen vorkommen. Eine solche Baseline „sichert sich ab“ und würde überschätzt (F1 39,5 statt 35,1). Durch das Zählen ganzer Sets ist die Vorhersage immer eine **real gewählte, klinisch plausible Kombination**, und die **Anzahl der Produkte ergibt sich aus den Daten**: Bei Experte 1 ist es 1 Produkt, bei Experte 2 sind es 2.

**Details:** Auf Level 2 werden die Sets vor dem Zählen auf Produktfamilien gemappt. Bei Gleichstand wird alphabetisch entschieden (deterministisch). In diesem Datensatz tritt das nicht auf, weil die häufigste Kombination jeweils deutlich vorne liegt.

In [3]:
from collections import Counter
for e in (1, 2):
    expert = load_expert_sets(e)
    combos = Counter(" + ".join(sorted(expert[w][0])) for w in evaluable_wounds(expert) if expert[w][0])
    print(f"Experte {e} – häufigste Präferenzkombinationen:")
    for combo, n in combos.most_common(4):
        print(f"   {n:2d}×  {combo}")
    print()

Experte 1 – häufigste Präferenzkombinationen:
   18×  Suprasorb CNP
    9×  Suprasorb P Sensiflex
    9×  Solvaline N + Suprasorb X + PHMB
    5×  Suprasorb Liquacel Pro + Suprasorb P Sensiflex + amorphes Gel

Experte 2 – häufigste Präferenzkombinationen:
   14×  Solvaline N + Suprasorb X + PHMB
    6×  Suprasorb Liquacel Pro + Suprasorb P Sensiflex
    6×  Suprasorb P Sensiflex
    3×  Solvaline N + Suprasorb P + PHMB



In [4]:
rows = []
for level in (1, 2):
    for e in (1, 2):
        r = majority_baseline_loo(e, level)
        rows.append({"Level": level, "Experte": e, "n": r["n"], "Majority F1 (%)": round(r["f1"], 1),
                     "Vorhersage (Häufigkeit über alle LOO-Durchgänge)": dict(r["gewaehlt"])})
pd.DataFrame(rows)

,Level,Experte,n,Majority F1 (%),Vorhersage (Häufigkeit über alle LOO-Durchgänge)
0,1,1,57,35.1,{'Suprasorb CNP': 57}
1,1,2,58,54.2,{'Solvaline N + Suprasorb X + PHMB': 58}
2,2,1,57,35.1,{'Suprasorb CNP (NPWT)': 57}
3,2,2,58,54.2,{'Solvaline (Folien-Saugkompresse) + Suprasorb...


## Ergebnisse im Überblick und Vergleich mit der KI
Zum Vergleich wird der KI-Score von Two-Stage (Originallauf) gegen denselben Experten mit `calculate_primaerverband_lr_expert_scores()` berechnet, also mit derselben Funktion wie in `03_primärverband.ipynb`.

In [5]:
table = compute_baseline_table(levels=(1, 2))
table["KI Two-Stage"] = [round(pv.calculate_primaerverband_lr_expert_scores(L, e)["right_values"][2], 1)
                         for L, e in zip(table["Level"], table["Experte"])]
table

,Level,Experte,n,Random,Majority (LOO),Majority-Vorhersage,KI Two-Stage
0,1,1,57,11.5,35.1,Suprasorb CNP,25.9
1,1,2,58,17.3,54.2,Solvaline N + Suprasorb X + PHMB,37.8
2,2,1,57,23.5,35.1,Suprasorb CNP (NPWT),47.1
3,2,2,58,35.1,54.2,Solvaline (Folien-Saugkompresse) + Suprasorb X...,67.2


## Hinweise zur Einordnung
- **Die Majority-Baseline kennt die Gewohnheiten des Experten.** Sie lernt aus den übrigen Wunden *desselben* Experten und kennt damit seinen „Hausstandard“ (z. B. Experte 2: *Solvaline N + Suprasorb X + PHMB*). Die KI hat nie eine Expertenentscheidung gesehen, sondern nur den Katalog. Dass Majority auf Level 1 über der KI liegt, zeigt vor allem, **wie stark die Produktwahl auf Produktebene von expertenindividuellen Gewohnheiten geprägt ist**.
- **Experte 1 und CNP:** Experte 1 empfiehlt bei 18 Wunden ausschließlich Suprasorb CNP (Unterdrucktherapie), die KI praktisch nie. Die Majority-Baseline „immer CNP“ trifft diese Wunden exakt.
- **Anzahl der Versuche:** Random wird wie die KI über Präferenz **und** Alternative bewertet, Majority nur mit einer Präferenz ohne Alternative.
- **Mittelwert über beide Experten:** Der Mittelwert der Einzelwerte pro Experte ist mit dem Mittelwert der KI-Scores pro Experte vergleichbar, **nicht** mit den Best-of-Both-Scores der KI. Best-of-Both nimmt pro Wunde das Maximum über beide Experten und liegt dadurch systematisch höher.
- **Abgrenzung:** Die fest eingetragenen Baselines in `primaerverband_calc.py` (z. B. Random 22,5 / 31,4, Majority 60,7 / 61,7) sind Best-of-Both-Werte mit anderer Definition und nicht mit diesen Werten vergleichbar. `eval/baselines.py` gehört zu einer separaten Auswertungspipeline.